# Test — Qwen3-Embedding vía vLLM

Notebook para validar embeddings con Qwen3-Embedding en dos modos:

1. **Online** — API OpenAI-compatible (`/v1/embeddings`) contra un servidor vLLM.
2. **Offline** — carga el modelo en proceso con `LLM(..., task="embed")`.

Las secciones 1 y 2 usan **los mismos** `queries`, `documents` e `input_texts` (ejemplo oficial de Qwen) y el mismo score `embeddings[:2] @ embeddings[2:].T`.

**Servidor online (secciones 1 y 3):**

```bash
VLLM_MODEL=Qwen/Qwen3-Embedding-8B VLLM_PORT=8001 ./scripts/vllm/serve_embedding_0.6b.sh
```

**Inferencia offline (sección 2):** detén el servidor antes de ejecutar la celda para liberar VRAM.

In [4]:
import os

import torch
from openai import OpenAI

VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://127.0.0.1:8001/v1")
VLLM_MODEL = os.getenv("VLLM_MODEL", "Qwen/Qwen3-Embedding-8B")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "EMPTY")

client = OpenAI(base_url=VLLM_BASE_URL, api_key=VLLM_API_KEY)

print("base_url:", VLLM_BASE_URL)
print("model:", VLLM_MODEL)

base_url: http://127.0.0.1:8001/v1
model: Qwen/Qwen3-Embedding-8B


In [5]:
def get_detailed_instruct(task_description: str, query: str) -> str:
    return f"Instruct: {task_description}\nQuery:{query}"


# Each query must come with a one-sentence instruction that describes the task
task = "Given a web search query, retrieve relevant passages that answer the query"

queries = [
    get_detailed_instruct(task, "What is the capital of China?"),
    get_detailed_instruct(task, "Explain gravity"),
]
# No need to add instruction for retrieval documents
documents = [
    "The capital of China is Beijing.",
    (
        "Gravity is a force that attracts two bodies towards each other. "
        "It gives weight to physical objects and is responsible for the "
        "movement of planets around the sun."
    ),
]
input_texts = queries + documents

print(f"input_texts: {len(queries)} queries + {len(documents)} documents")

inputs compartidos: 2 queries + 2 documents


## 1. Smoke test (online)

Mismo `input_texts` que la sección offline, vía API `/v1/embeddings`.

In [6]:
response = client.embeddings.create(model=VLLM_MODEL, input=input_texts)
embeddings = torch.tensor([item.embedding for item in response.data])
scores = embeddings[:2] @ embeddings[2:].T

print(scores.tolist())
# [[0.7482624650001526, 0.07556197047233582], [0.08875375241041183, 0.6300010681152344]]

dimensión del embedding: 4096
matriz de similitud (queries x documents):
[[0.7502 0.0757]
 [0.0888 0.6326]]


## 2. Inferencia offline (`LLM.embed`)

Mismo `input_texts` (ejemplo oficial de Qwen). **Detén el servidor online** antes de ejecutar esta celda si comparten la misma GPU.

In [7]:
# Requires vllm>=0.8.5
import vllm
from vllm import LLM

OFFLINE_MODEL = os.getenv("VLLM_OFFLINE_MODEL", VLLM_MODEL)

model = LLM(model=OFFLINE_MODEL, task="embed")
outputs = model.embed(input_texts)
embeddings = torch.tensor([o.outputs.embedding for o in outputs])
scores = embeddings[:2] @ embeddings[2:].T

print(scores.tolist())
# [[0.7482624650001526, 0.07556197047233582], [0.08875375241041183, 0.6300010681152344]]

/petrobr/home/ronaldinho.olivera/icl-embeddings/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 06-08 02:32:48 [__init__.py:216] Automatically detected platform cuda.
INFO 06-08 02:32:52 [utils.py:233] non-default args: {'task': 'embed', 'disable_log_stats': True, 'model': 'Qwen/Qwen3-Embedding-8B'}
INFO 06-08 02:32:56 [model.py:547] Resolved architecture: Qwen3ForCausalLM
INFO 06-08 02:32:56 [config.py:739] Found sentence-transformers modules configuration.
INFO 06-08 02:32:56 [config.py:759] Found pooling configuration.


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 06-08 02:32:56 [model.py:1510] Using max model len 40960
INFO 06-08 02:32:56 [arg_utils.py:1575] (Enabling) chunked prefill by default
INFO 06-08 02:32:56 [arg_utils.py:1578] (Enabling) prefix caching by default


2026-06-08 02:32:56,925	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


...

(EngineCore_DP0 pid=2569436) INFO 06-08 02:33:13 [default_loader.py:267] Loading weights took 6.99 seconds
(EngineCore_DP0 pid=2569436) INFO 06-08 02:33:13 [gpu_model_runner.py:2653] Model loading took 14.1062 GiB and 11.246646 seconds
(EngineCore_DP0 pid=2569436) INFO 06-08 02:33:18 [backends.py:548] Using cache directory: /petrobr/home/ronaldinho.olivera/.cache/vllm/torch_compile_cache/5124f6434a/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=2569436) INFO 06-08 02:33:18 [backends.py:559] Dynamo bytecode transform time: 4.83 s
(EngineCore_DP0 pid=2569436) INFO 06-08 02:33:21 [backends.py:164] Directly load the compiled graph(s) for dynamic shape from the cache, took 2.180 s
(EngineCore_DP0 pid=2569436) INFO 06-08 02:33:22 [monitor.py:34] torch.compile takes 4.83 s in total
(EngineCore_DP0 pid=2569436) INFO 06-08 02:33:23 [gpu_worker.py:298] Available KV cache memory: 55.39 GiB
(EngineCore_DP0 pid=2569436) INFO 06-08 02:33:24 [kv_cache_utils.py:1087] GPU KV cache size:

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:02<00:00, 25.12it/s]


(EngineCore_DP0 pid=2569436) INFO 06-08 02:33:27 [gpu_model_runner.py:3480] Graph capturing finished in 4 secs, took 0.70 GiB
(EngineCore_DP0 pid=2569436) INFO 06-08 02:33:27 [core.py:210] init engine (profile, create kv cache, warmup model) took 13.93 seconds
INFO 06-08 02:33:29 [llm.py:306] Supported_tasks: ['embed']


Processed prompts: 100%|██████████| 4/4 [00:00<00:00, 153.85it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

model: Qwen/Qwen3-Embedding-8B
dimensión del embedding: 4096
matriz de similitud (queries x documents):
[[0.7502 0.0757]
 [0.0888 0.6326]]


## 3. Ejemplo FiQA (online)

Carga una consulta con sus pasajes relevantes del benchmark FiQA y comprueba si el documento gold obtiene mayor similitud que un pasaje aleatorio del corpus.

In [ ]:
import random

from icl_embeddings.datasets import get_beir_dataset, passage_text

corpus, fiqa_queries, qrels = get_beir_dataset("fiqa")

qid = next(iter(fiqa_queries))
query_text = fiqa_queries[qid]
positive_ids = [doc_id for doc_id, score in qrels[qid].items() if score > 0]

random_neg_id = random.choice(
    [doc_id for doc_id in corpus if doc_id not in positive_ids]
)

candidate_docs = {
    "gold": passage_text(corpus[positive_ids[0]]),
    "random": passage_text(corpus[random_neg_id]),
}

print(f"query (id={qid!r}):", query_text)
print("\n--- gold ---")
print(candidate_docs["gold"][:400], "...")
print("\n--- random ---")
print(candidate_docs["random"][:400], "...")

In [ ]:
fiqa_query = get_detailed_instruct(task, query_text)
fiqa_docs = list(candidate_docs.values())
fiqa_labels = list(candidate_docs.keys())

fiqa_input_texts = [fiqa_query] + fiqa_docs
response = client.embeddings.create(model=VLLM_MODEL, input=fiqa_input_texts)
embeddings = torch.tensor([item.embedding for item in response.data])
fiqa_scores = (embeddings[:1] @ embeddings[1:].T).flatten()

for label, score in sorted(
    zip(fiqa_labels, fiqa_scores), key=lambda x: x[1], reverse=True
):
    print(f"{label:>6}: {score:.4f}")

gold_wins = fiqa_scores[fiqa_labels.index("gold")] > fiqa_scores[
    fiqa_labels.index("random")
]
print("\n¿Gold > random?", gold_wins)